# rtp-audit pilot on Colab (H200 141 GB or 96 GB GPU)

**Open this notebook straight from GitHub:** [Open in Colab](https://colab.research.google.com/github/GuangLun2000/RTP-Audit/blob/claude/rtp-audit-m0-m2-824c69/colab/run_pilot.ipynb). Colab then loads the newest pushed version every time; there is no copy to keep on Drive (if you do save a copy, cell 2 warns when it is outdated).

- **Code** comes from GitHub: cell 2 clones the repository into `/content/rtp-audit` (local disk) and updates it to the newest push of the pinned branch.
- **Results** go to `RTP_WORK_DIR`. With `USE_DRIVE = True` (cell 1) that is `MyDrive/rtp` on Google Drive, so finished shards survive a disconnect; with `False` everything stays on the VM and is lost when the runtime resets.
- **Before running:** add `HF_TOKEN` in Colab *Secrets* (accept the Llama 3.1 licence on the Hub first).

How this notebook is meant to be used:
1. Run the cells top to bottom. Every stage prints live progress (one line per ~10% with ETA), marks warnings ⚠️ and errors ❌, and records them in `runs/<config>/events.jsonl`.
2. Every stage is resumable: after a disconnect or runtime reset, re-run cells 1-3, then the cell that was interrupted. Finished shards and finished stages are reused when their code and inputs are unchanged.
3. When something fails or looks wrong, run the **last cell**. It writes a *feedback bundle* (environment, stage status, errors with tracebacks, warnings, key results, stage summaries, suggested next actions). Give that file to the coding agent ([docs/AGENT_LOOP.md](../docs/AGENT_LOOP.md)); after the fix is pushed, re-run cell 2 (it pulls the fix) and the failed cell: only stages whose code or inputs changed are recomputed.


In [ ]:
# Cell 1: settings, then results storage and environment variables  (spec §13, cell 1)
NOTEBOOK_REV = "6a9288915b"  # identifies this notebook's code cells; cell 2 compares it with the repo's copy
REPO_URL = "https://github.com/GuangLun2000/RTP-Audit.git"
PIN = "claude/rtp-audit-m0-m2-824c69"  # a branch (updated to its newest push) or a commit hash (exactly that code)
USE_DRIVE = True                       # results on Drive survive disconnects; False = /content/rtp_runs (lost on reset)
CONFIG = "configs/pilot.yaml"
MODELS = [
    "allenai/Llama-3.1-Tulu-3-8B-SFT",
    "allenai/Llama-3.1-Tulu-3-8B-DPO",
]

import os
from google.colab import userdata

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ["RTP_WORK_DIR"] = "/content/drive/MyDrive/rtp"
else:
    os.environ["RTP_WORK_DIR"] = "/content/rtp_runs"
os.environ["HF_HOME"] = "/content/hf"          # model cache on local disk, not Drive
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:                        # missing secret: gated Llama repos will fail, others work
    print("⚠️ HF_TOKEN secret not available:", type(exc).__name__)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["RTP_PROGRESS"] = "log"              # readable progress lines instead of carriage-return bars
os.makedirs(os.environ["RTP_WORK_DIR"], exist_ok=True)
print("results in", os.environ["RTP_WORK_DIR"])


In [ ]:
# Cell 2: get the code from GitHub and install it  (spec §13, cell 2; torch is never reinstalled)
# Stops with ❌ and a reason at the first failing step; ends with "✅ code ready" when everything worked.
import importlib, json, os, re, subprocess, sys

REPO_DIR = "/content/rtp-audit"
OPEN_LINK = "https://colab.research.google.com/github/GuangLun2000/RTP-Audit/blob/claude/rtp-audit-m0-m2-824c69/colab/run_pilot.ipynb"


def sh(*cmd, cwd=None, check=True):
    try:
        r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    except FileNotFoundError:
        if check:
            raise RuntimeError(f"❌ program not found: {cmd[0]}")
        print("$", " ".join(cmd), "\n(not available)")
        return None
    out = (r.stdout + r.stderr).strip()
    print("$", " ".join(cmd) + ("" if not out else "\n" + out[-3000:]))
    if check and r.returncode:
        raise RuntimeError(f"❌ step failed (exit {r.returncode}): {' '.join(cmd)}")
    return r


sh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv", check=False)
if not str(REPO_URL).startswith("https://"):
    raise RuntimeError(f"❌ REPO_URL is not set: this is an old copy of the notebook. Open it from GitHub: {OPEN_LINK}")
if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh("git", "clone", "-q", REPO_URL, REPO_DIR)
sh("git", "fetch", "-q", "--all", "--tags", cwd=REPO_DIR)
sh("git", "checkout", "-q", PIN, cwd=REPO_DIR)
if sh("git", "symbolic-ref", "-q", "HEAD", cwd=REPO_DIR, check=False).returncode == 0:  # a branch: newest push
    sh("git", "pull", "-q", "--ff-only", cwd=REPO_DIR)
os.chdir(REPO_DIR)

import torch

torch_before = torch.__version__
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]", cwd=REPO_DIR)
ver = sh(sys.executable, "-c", "import torch, transformers, rtp; print(torch.__version__, transformers.__version__)",
         cwd=REPO_DIR).stdout.split()
if ver[0] != torch_before:
    raise RuntimeError(f"❌ torch changed from {torch_before} to {ver[0]}; restart the runtime (never reinstall torch)")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
repo_nb = json.load(open(f"{REPO_DIR}/colab/run_pilot.ipynb", encoding="utf-8"))
m = re.search(r'(?m)^NOTEBOOK_REV = "([^"]+)"', "".join("".join(c["source"]) for c in repo_nb["cells"]))
if m and m.group(1) != NOTEBOOK_REV:
    print(f"⚠️ this notebook (rev {NOTEBOOK_REV}) differs from the repo's (rev {m.group(1)}). Reopen it from GitHub: "
          f"{OPEN_LINK}")
head = sh("git", "log", "-1", "--format=%h %s", cwd=REPO_DIR).stdout.strip()
print(f"✅ code ready: {head} | torch {ver[0]} (unchanged) | transformers {ver[1]}")


In [ ]:
# Cell 3: preflight. GPU, bf16, library pins, HF token, disks, Drive, data, Hub access to MODELS
import importlib

try:
    import rtp.notebook as nb
except ModuleNotFoundError as exc:
    raise RuntimeError("❌ the rtp package is not importable: run cell 2 first; it must end with '✅ code ready'") from exc
importlib.reload(nb)          # pick up helper changes after a `git pull` without restarting the runtime
nb.run("doctor", "--config", CONFIG)
OVR = nb.gpu_overrides()      # batch sizes for this GPU (never part of a stage's identity: safe to change)
print("batch overrides:", OVR)


In [ ]:
# Cell 4: smoke test and the test suite (a few minutes)  (spec §13, cell 3)
nb.run("smoke")
nb.pytest(CONFIG)             # RTP_REQUIRE_INTEGRATION=1: on Colab the smoke model must load, skips count as failures

In [ ]:
# Cell 5: generate the data (only once; reused afterwards)  (spec §13, cell 4)
nb.run("build-data", "--config", CONFIG, strict=True)
nb.show(f"{os.environ['RTP_WORK_DIR']}/runs/pilot/data/data_report.md", max_chars=4000)

In [ ]:
# Cell 5b: answer-format probe (a few minutes per model). Look at it BEFORE the long runs: low c_forced means a
# template / prompt-mode problem; many 'deferred' answers mean O5 leans on the imputation (see DECISIONS.md)
for m in MODELS:
    nb.run(*OVR, "format-probe", "--config", CONFIG, "--model", m)
    nb.show(f"{os.environ['RTP_WORK_DIR']}/runs/pilot/format_probe/{m.replace('/', '__')}/summary.md", max_chars=5000)


In [ ]:
# Cell 6: behavioural measurement + M3 activations/probes/localisation; resumable, re-run after an interruption  (spec §13, cell 5)
for m in MODELS:
    for stage in ("measure", "open-ended", "extract"):
        nb.run(*OVR, stage, "--config", CONFIG, "--model", m)   # a failure is recorded; the loop continues
    nb.run("probe", "--config", CONFIG, "--model", m)
    nb.run(*OVR, "localize", "--config", CONFIG, "--model", m)   # where the name effect is carried
    nb.run("status", "--config", CONFIG)

In [ ]:
# Cell 7: summaries  (spec §13, cell 6)
nb.run("report", "--config", CONFIG)
RUN = f"{os.environ['RTP_WORK_DIR']}/runs/pilot"
nb.show(f"{RUN}/report.md")
for m in MODELS:
    slug = m.replace("/", "__")
    nb.show(f"{RUN}/{slug}/probe/summary.md", max_chars=6000)   # M4 readiness warnings
    nb.show(f"{RUN}/{slug}/localize/summary.md", max_chars=6000)  # transplant recovery by layer/site

In [ ]:
# Cell 8: feedback bundle for the coding agent (run this whenever something failed or looks wrong)
bundle = nb.feedback(CONFIG)
if bundle:
    nb.show(bundle, max_chars=8000)
    from google.colab import files
    files.download(str(bundle))   # or open it on Drive: runs/pilot/feedback/